# Notebook 5: Production Ingestion Framework & Final Capstone Project (Student Lab)
### Hands-on Workshop: Apache Spark Foundation & Ingestion Framework (Day 2 Finale)
### Related Presentation Slides: Slides 26 - 28 (Module 8 & Final Project)

---

## Learning Objectives:
1. Review production engineering standards: Structured JSON Logging and Custom Exceptions (Slide 26).
2. Run the complete end-to-end framework.
3. Complete the Final Capstone Assignment (Slide 27).


In [ ]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from dataclasses import dataclass
import json, logging, sys

spark = SparkSession.builder \
    .appName("05_Production_Framework_Capstone") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Ready.")


---
## Step 1: Enterprise Logging & Exception Hierarchy (Related: Slide 26)

Production pipelines avoid plain `print()` statements.
They emit structured JSON logs ingested by CloudWatch, Datadog, or ELK stacks (Slide 26).


In [ ]:
class StructuredLogger:
    def __init__(self, app_name: str):
        self.app_name = app_name

    def log(self, level: str, message: str, **kwargs):
        payload = {
            "application": self.app_name,
            "level": level.upper(),
            "message": message,
            "extra": kwargs
        }
        print(json.dumps(payload))

logger = StructuredLogger("BundesligaIngestionFramework")
logger.log("INFO", "Framework initialized", version="1.0.0", environment="Colab_Cloud")

# Custom Exceptions (Slide 26)
class FrameworkError(Exception): pass
class DataIngestionError(FrameworkError): pass
class DataWriterError(FrameworkError): pass

print("Logging & Exceptions defined.")


---
## Step 2: End-to-End Pipeline Execution
Assemble Reader, Transformer, and Writer into one clean orchestration function:


In [ ]:
def run_full_pipeline(spark_session: SparkSession, raw_json_path: str, output_parquet_path: str):
    logger.log("INFO", "Starting pipeline run", source=raw_json_path)
    
    # 1. Read
    df_raw = spark_session.read.json(raw_json_path)
    logger.log("INFO", f"Raw records ingested: {df_raw.count()}")
    
    # 2. Transform
    df_transformed = df_raw.select(
        F.coalesce(F.col("event_id"), F.col("id")).alias("event_id"),
        F.col("team.name").alias("team_name"),
        F.col("player.name").alias("player_name"),
        F.col("minute"),
        F.col("type.name").alias("event_type"),
        F.col("shot.statsbomb_xg").alias("xg"),
        F.coalesce(F.col("shot.outcome.name"), F.col("shot.outcome")).alias("outcome")
    ).withColumn(
        "is_goal", F.when(F.col("outcome") == "Goal", 1).otherwise(0)
    )
    
    # 3. Write
    logger.log("INFO", "Writing output Parquet partitioned by team_name", target=output_parquet_path)
    df_transformed.coalesce(2) \
        .write \
        .mode("overwrite") \
        .partitionBy("team_name") \
        .parquet(output_parquet_path)
        
    logger.log("INFO", "Pipeline completed successfully")
    return df_transformed

df_final = run_full_pipeline(spark, "data/raw/bundesliga_events.json", "data/processed/capstone_output")
df_final.show(5)


---
## Final Capstone Project Assignment (Related: Slide 27)

### Business Context:
The Analytics department requires an automated daily report analyzing player offensive efficiency across the Bundesliga.

### Requirements (Slide 27):
1. Filter: Keep only events where `event_type == 'Shot'`.
2. Aggregations: Group by `player_name` and `team_name` to calculate:
   * `total_shots` = count of shots
   * `total_goals` = sum of `is_goal`
   * `total_xg` = sum of `xg` rounded to 2 decimal places
   * `shot_conversion_pct` = `(total_goals / total_shots) * 100` rounded to 1 decimal place
3. Enrichment (Broadcast Join):
   * Join with `data/raw/team_financials.csv` using a Broadcast Join on `team_name`.
   * Include the column `budget_eur_millions`.
4. Partitioned Export:
   * Coalesce the resulting DataFrame to 1 partition.
   * Save the result as Parquet in `data/processed/player_efficiency_kpi` partitioned by `team_name`.


### Participant Implementation Cell:


In [ ]:
# TODO: Implement your Capstone Solution here:

# Step 1: Read events and financials
# df_events = ...
# df_financials = ...

# Step 2: Calculate player aggregates
# df_kpi = ...

# Step 3: Broadcast Join with financials
# df_enriched = ...

# Step 4: Coalesce and write to Parquet
# ...


In [ ]:
# --- VERIFICATION TEST SUITE (Run this cell to validate your submission) ---
try:
    test_df = spark.read.parquet("data/processed/player_efficiency_kpi")
    print(f"Total rows in Capstone output: {test_df.count()}")
    print("Columns verified:", test_df.columns)
    test_df.show(10)
    print("Capstone Project successfully completed.")
except Exception as e:
    print(f"Verification failed: {e}")
